In [ ]:
#1.1 Modo 'b' (binario)

#Sirve para archivos que no son texto (imágenes, PDFs, audios) 
#y para guardar objetos de Python con pickle.



In [ ]:
# ARCHIVOS BINARIOS (modo 'b') y pickle

# 'b' se junta con otro modo:
#   'rb' = leer binario    'wb' = escribir binario    'ab' = añadir binario
# En binario NO se escribe texto (str), se escriben bytes.
# Un bytes se escribe con una b delante de las comillas: b"Hola"
# Se usa para imágenes, audios, PDFs... y para guardar objetos de Python.

import pickle   # librería para guardar objetos enteros en un archivo

# ----- Ejemplo: escribir y leer bytes -----
datos = bytes([72, 111, 108, 97])       # crea bytes desde una lista de números (0 a 255)
with open("datos.bin", "wb") as f:      # wb = write binary
    f.write(datos)

with open("datos.bin", "rb") as f:      # rb = read binary
    leido = f.read()                    # read() devuelve un objeto bytes
print(leido)                            # b'Hola'
print(list(leido))                      # [72, 111, 108, 97]  (cada byte es un número)
print(leido.decode("utf-8"))            # Hola  (decode: bytes -> texto)
print("Adiós".encode("utf-8"))          # b'Adi\xc3\xb3s'  (encode: texto -> bytes)


# ----- EJERCICIO 1: copiar un archivo cualquiera (imagen, pdf...) en binario -----
def copiar_archivo(origen, destino):
    # Se pueden abrir dos archivos en el mismo with, separados por coma
    with open(origen, "rb") as entrada, open(destino, "wb") as salida:
        while True:
            trozo = entrada.read(1024)    # lee un trozo de 1024 bytes
            if not trozo:                 # al final del archivo read() devuelve b"" (vacío)
                break                     # salimos del bucle
            salida.write(trozo)           # escribe el trozo en la copia


copiar_archivo("datos.bin", "copia.bin")
with open("copia.bin", "rb") as f:
    print(f.read())                       # b'Hola'  -> la copia es idéntica


# ----- EJERCICIO 2: guardar y recuperar objetos con pickle -----
class Alumno:
    def __init__(self, nombre, nota):
        self.nombre = nombre
        self.nota = nota


alumnos = [Alumno("Ana", 8.5), Alumno("Luis", 6.0)]

with open("alumnos.dat", "wb") as f:
    pickle.dump(alumnos, f)               # dump: guarda la lista de objetos entera

with open("alumnos.dat", "rb") as f:
    recuperados = pickle.load(f)          # load: la recupera tal cual estaba

for a in recuperados:
    print(a.nombre, a.nota)               # Ana 8.5  /  Luis 6.0

In [ ]:
#1.2 Modos con '+' (leer y escribir a la vez)

In [ ]:
# MODOS CON '+' (leer y escribir a la vez)

# 'r+' : leer Y escribir. El archivo tiene que existir. NO borra el contenido.
#        El cursor empieza al principio, así que lo que escribas pisa lo que había.
# 'w+' : leer Y escribir, pero BORRA todo al abrir (como 'w').
# 'a+' : leer Y añadir al final. Para leer hay que volver al principio con seek(0).
#
# El "cursor" es la posición por la que va el archivo:
#   f.tell()  -> dice en qué posición está el cursor
#   f.seek(0) -> lleva el cursor al principio (seek(n) a la posición n)

with open("frase.txt", "w") as f:
    f.write("Hola mundo\n")

# ----- r+ -----
with open("frase.txt", "r+") as f:
    print(f.read(4))       # Hola   (read(4) lee solo 4 caracteres)
    print(f.tell())        # 4      (el cursor está en la posición 4)
    f.seek(0)              # volvemos al principio
    f.write("HOLA")        # pisa los 4 primeros caracteres
    f.seek(0)              # volvemos al principio para leer todo
    print(f.read())        # HOLA mundo

# ----- w+ -----
with open("nuevo.txt", "w+") as f:     # crea el archivo vacío
    f.write("Uno\nDos\n")
    f.seek(0)                          # sin esto read() no leería nada (cursor al final)
    print(f.read())                    # Uno / Dos

# ----- a+ -----
with open("nuevo.txt", "a+") as f:
    f.write("Tres\n")                  # siempre se escribe al final
    f.seek(0)
    print(f.read())                    # Uno / Dos / Tres


# ----- EJERCICIO: contador de visitas guardado en un archivo -----
# El archivo guarda un solo número. Cada vez que se llama, se lee, se suma 1 y se vuelve a guardar.
def contar_visita(archivo):
    try:
        f = open(archivo, "r+")        # intenta abrirlo sin borrar
    except FileNotFoundError:
        f = open(archivo, "w+")        # si no existe, lo crea
        f.write("0")                   # empieza en 0
        f.seek(0)
    with f:                            # with cierra el archivo automáticamente al salir
        visitas = int(f.read()) + 1    # leemos el número y sumamos 1
        f.seek(0)                      # volvemos al principio
        f.truncate()                   # truncate() borra todo lo que hay desde el cursor
        f.write(str(visitas))          # guardamos el nuevo número
    return visitas


print(contar_visita("visitas.txt"))    # 1
print(contar_visita("visitas.txt"))    # 2
print(contar_visita("visitas.txt"))    # 3

In [ ]:
#1.3 Excepciones (try / except)

In [ ]:
# EXCEPCIONES (try / except)

# Una excepción es un error que ocurre mientras el programa se ejecuta.
# Si no la controlas, el programa se para. Con try/except la controlas:
#
#   try:       -> aquí va el código que puede fallar
#   except X:  -> se ejecuta SOLO si ocurre el error X
#   else:      -> se ejecuta si NO ha habido ningún error
#   finally:   -> se ejecuta SIEMPRE, haya error o no (sirve para limpiar, cerrar...)
#
# Errores típicos:
#   FileNotFoundError  -> el archivo no existe
#   ValueError         -> valor incorrecto, por ejemplo int("abc")
#   ZeroDivisionError  -> dividir entre 0
#   TypeError          -> tipo incorrecto, por ejemplo "a" + 1
#   IndexError         -> posición que no existe en una lista
#   KeyError           -> clave que no existe en un diccionario

# ----- Ejemplo básico -----
try:
    numero = int("abc")                    # esto falla: "abc" no es un número
    print("Esto no se ejecuta")
except ValueError:
    print("Eso no es un número")           # se ejecuta esto

# ----- Guardar el mensaje del error con "as" -----
try:
    print(10 / 0)
except ZeroDivisionError as error:
    print("Error:", error)                 # Error: division by zero


# ----- EJERCICIO 1: leer números de un archivo ignorando las líneas que fallen -----
with open("numeros.txt", "w") as f:
    f.write("10\n20\nabc\n30\n\n")         # una línea mala ("abc") y una vacía


def leer_numeros(archivo):
    numeros = []
    try:
        with open(archivo, "r") as f:
            # enumerate numera las líneas: n vale 1, 2, 3... y linea es el texto
            for n, linea in enumerate(f, start=1):
                linea = linea.strip()                 # quita espacios y \n
                if linea == "":
                    continue                          # continue salta a la siguiente vuelta
                try:
                    numeros.append(int(linea))        # intenta convertirla
                except ValueError:
                    print(f"Línea {n}: '{linea}' no es un número, se ignora")
    except FileNotFoundError:
        print(f"El archivo {archivo} no existe")
    return numeros


print(leer_numeros("numeros.txt"))         # Línea 3 ... y luego [10, 20, 30]
print(leer_numeros("no_existe.txt"))       # El archivo no_existe.txt no existe  y  []


# ----- EJERCICIO 2: else y finally -----
def dividir(a, b):
    try:
        resultado = a / b
    except ZeroDivisionError:
        print("No se puede dividir entre 0")
        return None
    except TypeError:
        print("Hay que pasar números")
        return None
    else:
        print("Todo ha ido bien")                     # solo si no hubo error
        return resultado
    finally:
        print("Fin de dividir")                       # se ejecuta SIEMPRE


print(dividir(10, 2))     # Todo ha ido bien / Fin de dividir / 5.0
print(dividir(10, 0))     # No se puede dividir entre 0 / Fin de dividir / None
print(dividir(10, "a"))   # Hay que pasar números / Fin de dividir / None


# ----- EJERCICIO 3: crear tu propia excepción con raise -----
# Una excepción propia es una clase que hereda de Exception
class SaldoInsuficiente(Exception):
    pass


class Cuenta:
    def __init__(self, saldo=0):
        self.saldo = saldo

    def retirar(self, cantidad):
        if cantidad > self.saldo:
            # raise "lanza" la excepción a propósito
            raise SaldoInsuficiente(f"Tienes {self.saldo} y quieres retirar {cantidad}")
        self.saldo -= cantidad


cuenta = Cuenta(50)
try:
    cuenta.retirar(80)
except SaldoInsuficiente as error:
    print("Error:", error)                # Error: Tienes 50 y quieres retirar 80